In [6]:
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
import krippendorff
from scipy.stats import pearsonr
from math import sqrt
from glob import glob
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [7]:

# -----------------------
# Paths
# -----------------------
anno_path = r"D:\\Downloads\\DL_Assignment1_Dataset\\Dataset\\Dataset\\annotations"
img_path  = r"D:\\Downloads\\DL_Assignment1_Dataset\\Dataset\\Dataset\\images"

# -----------------------
# Step 1: Find available annotation indices
# -----------------------
exp_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_exp.npy"))}
val_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_val.npy"))}
aro_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_aro.npy"))}
lnd_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_lnd.npy"))}

# Only keep indices that have ALL 4 annotations
valid_indices = sorted(list(exp_files & val_files & aro_files & lnd_files), key=int)

print(f"Total annotated samples: {len(valid_indices)}")

# -----------------------
# Step 2: Load annotations + images
# -----------------------
images, expressions, valence, arousal, landmarks = [], [], [], [], []

for idx in valid_indices:
    e_file = os.path.join(anno_path, f"{idx}_exp.npy")
    v_file = os.path.join(anno_path, f"{idx}_val.npy")
    a_file = os.path.join(anno_path, f"{idx}_aro.npy")
    l_file = os.path.join(anno_path, f"{idx}_lnd.npy")
    img_file = os.path.join(img_path, f"{idx}.jpg")  # adjust extension if .png
    
    # skip if image file not found
    if not os.path.exists(img_file):
        continue
    
    # Load annotations
    exp = np.load(e_file).squeeze()
    val = np.load(v_file).squeeze()
    aro = np.load(a_file).squeeze()
    lnd = np.load(l_file)

    # Skip invalid valence/arousal = -2
    if val == -2 or aro == -2:
        continue

    # Load image
    img = load_img(img_file, target_size=(224, 224))
    img = img_to_array(img) / 255.0

    # Append
    images.append(img)
    expressions.append(exp)
    valence.append(val)
    arousal.append(aro)
    landmarks.append(lnd)

# Convert to arrays
images = np.array(images, dtype="float32")
expressions = np.array(expressions)
valence = np.array(valence)
arousal = np.array(arousal)
landmarks = np.array(landmarks)

print("Final dataset shapes:")
print("Images:", images.shape)
print("Expressions:", expressions.shape)
print("Valence:", valence.shape)
print("Arousal:", arousal.shape)
print("Landmarks:", landmarks.shape)

# -----------------------
# Step 3: Train/Val/Test split
# -----------------------
X_train, X_test, y_train_exp, y_test_exp, y_train_val, y_test_val, y_train_aro, y_test_aro = train_test_split(
    images, expressions, valence, arousal, test_size=0.2, random_state=42, stratify=expressions
)

X_train, X_val, y_train_exp, y_val_exp, y_train_val, y_val_val, y_train_aro, y_val_aro = train_test_split(
    X_train, y_train_exp, y_train_val, y_train_aro, test_size=0.2, random_state=42, stratify=y_train_exp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


Total annotated samples: 3999
Final dataset shapes:
Images: (3999, 224, 224, 3)
Expressions: (3999,)
Valence: (3999,)
Arousal: (3999,)
Landmarks: (3999, 136)
Train: (2559, 224, 224, 3) Val: (640, 224, 224, 3) Test: (800, 224, 224, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from scipy.stats import pearsonr
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import pandas as pd

# -----------------------
# Paths
# -----------------------
anno_path = r"D:\\Downloads\\DL_Assignment1_Dataset\\Dataset\\Dataset\\annotations"
img_path  = r"D:\\Downloads\\DL_Assignment1_Dataset\\Dataset\\Dataset\\images"

# -----------------------
# Step 1: Find available annotation indices (MEMORY EFFICIENT)
# -----------------------
exp_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_exp.npy"))}
val_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_val.npy"))}
aro_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_aro.npy"))}
lnd_files = {os.path.basename(f).split("_")[0] for f in glob(os.path.join(anno_path, "*_lnd.npy"))}

# Only keep indices that have ALL 4 annotations
valid_indices = sorted(list(exp_files & val_files & aro_files & lnd_files), key=int)
print(f"Total annotated samples: {len(valid_indices)}")

# -----------------------
# Step 2: Create metadata dataframe (NO IMAGES LOADED)
# -----------------------
metadata = []

for idx in valid_indices:
    e_file = os.path.join(anno_path, f"{idx}_exp.npy")
    v_file = os.path.join(anno_path, f"{idx}_val.npy")
    a_file = os.path.join(anno_path, f"{idx}_aro.npy")
    l_file = os.path.join(anno_path, f"{idx}_lnd.npy")
    img_file = os.path.join(img_path, f"{idx}.jpg")
    
    # Skip if image file not found
    if not os.path.exists(img_file):
        continue
    
    # Load annotations only (not images)
    exp = np.load(e_file).squeeze()
    val = np.load(v_file).squeeze()
    aro = np.load(a_file).squeeze()
    lnd = np.load(l_file)

    # Skip invalid valence/arousal = -2
    if val == -2 or aro == -2:
        continue

    metadata.append({
        'image_path': img_file,
        'expression': exp,
        'valence': val,
        'arousal': aro,
        'landmarks': lnd
    })

# Convert to DataFrame
metadata_df = pd.DataFrame(metadata)
print(f"Final dataset size: {len(metadata_df)} samples")

# Check class distribution
print("Class distribution:")
print(metadata_df['expression'].value_counts().sort_index())

# -----------------------
# Step 3: Train/Val/Test split (ON METADATA)
# -----------------------
train_df, test_df = train_test_split(
    metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['expression']
)

train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['expression']
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# -----------------------
# Step 4: Custom Data Generator (MEMORY EFFICIENT)
# -----------------------
class FacialExpressionGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=32, shuffle=True, augment=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.on_epoch_end()
        
        # Data augmentation
        self.augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
        ])
        
    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))
    
    def __getitem__(self, index):
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, len(self.dataframe))
        
        batch_data = self.dataframe.iloc[start_idx:end_idx]
        
        # Initialize batches
        batch_images = []
        batch_expressions = []
        batch_va = []
        
        for _, row in batch_data.iterrows():
            # Load and preprocess image
            img = load_img(row['image_path'], target_size=(224, 224))
            img = img_to_array(img) / 255.0
            
            if self.augment:
                img = self.augmentation(tf.expand_dims(img, 0))[0]
            
            batch_images.append(img)
            batch_expressions.append(row['expression'])
            batch_va.append([row['valence'], row['arousal']])
        
        # Convert to arrays
        batch_images = np.array(batch_images, dtype="float32")
        batch_expressions = tf.keras.utils.to_categorical(batch_expressions, num_classes=8)
        batch_va = np.array(batch_va, dtype="float32")
        
        return batch_images, {
            "class_out": batch_expressions,
            "va_out": batch_va
        }
    
    def on_epoch_end(self):
        if self.shuffle:
            self.dataframe = self.dataframe.sample(frac=1).reset_index(drop=True)

# -----------------------
# Step 5: Create generators
# -----------------------
BATCH_SIZE = 32

train_generator = FacialExpressionGenerator(train_df, batch_size=BATCH_SIZE, shuffle=True, augment=True)
val_generator = FacialExpressionGenerator(val_df, batch_size=BATCH_SIZE, shuffle=False, augment=False)
test_generator = FacialExpressionGenerator(test_df, batch_size=BATCH_SIZE, shuffle=False, augment=False)

print("Generators created successfully!")
print(f"Train batches: {len(train_generator)}, Val batches: {len(val_generator)}")

# -----------------------
# Step 6: Model Building (UNCHANGED)
# -----------------------
def build_enhanced_multitask(backbone_name="ResNet50", input_shape=(224,224,3)):
    """Build enhanced multi-task model with unfrozen backbone"""
    
    inp = layers.Input(shape=input_shape)
    
    if backbone_name == "ResNet50":
        base = tf.keras.applications.ResNet50(
            include_top=False, weights="imagenet", input_tensor=inp, pooling="avg")
    elif backbone_name == "EfficientNetB0":
        base = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_tensor=inp, pooling="avg")
    else:
        raise ValueError("Unsupported backbone")
    
    base.trainable = True  # Unfrozen for feature learning
    feat = base.output

    # Enhanced feature processing
    x = layers.Dense(512, activation="relu")(feat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    # Expression classification head
    class_head = layers.Dense(256, activation="relu")(x)
    class_head = layers.Dropout(0.3)(class_head)
    class_out = layers.Dense(8, activation="softmax", name="class_out")(class_head)  # Fixed to 8 classes

    # Valence-Arousal regression head
    reg_head = layers.Dense(128, activation="relu")(x)
    reg_head = layers.Dropout(0.2)(reg_head)
    va_out = layers.Dense(2, activation="tanh", name="va_out")(reg_head)

    return models.Model(inputs=inp, outputs=[class_out, va_out])

# -----------------------
# Step 7: Simplified Metrics
# -----------------------
def ccc_tensor(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    true_mean = tf.reduce_mean(y_true, axis=0)
    pred_mean = tf.reduce_mean(y_pred, axis=0)
    var_true = tf.reduce_mean(tf.square(y_true - true_mean), axis=0)
    var_pred = tf.reduce_mean(tf.square(y_pred - pred_mean), axis=0)
    cov = tf.reduce_mean((y_true - true_mean) * (y_pred - pred_mean), axis=0)
    ccc = (2.0 * cov) / (var_true + var_pred + tf.square(true_mean - pred_mean) + 1e-8)
    return tf.reduce_mean(ccc)

def sagr_tensor(y_true, y_pred):
    sign_true = tf.sign(y_true)
    sign_pred = tf.sign(y_pred)
    agree = tf.cast(tf.equal(sign_true, sign_pred), tf.float32)
    return tf.reduce_mean(agree)

# -----------------------
# Step 8: Training Function
# -----------------------
def compile_and_train(model, train_gen, val_gen, exp_name, epochs=20, lr=1e-4):
    """Compile and train with simplified loss functions"""
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss={
            "class_out": "categorical_crossentropy",
            "va_out": "mse"
        },
        loss_weights={"class_out": 1.0, "va_out": 0.5},
        metrics={
            "class_out": ["accuracy"],
            "va_out": [ccc_tensor, sagr_tensor]
        }
    )

    # Callbacks
    ckpt = ModelCheckpoint(f"{exp_name}_best.h5", monitor="val_loss", 
                          save_best_only=True, mode="min", verbose=1)
    early = EarlyStopping(monitor="val_loss", patience=10, mode="min", 
                         restore_best_weights=True, verbose=1)
    reduce = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, 
                              min_lr=1e-7, verbose=1)

    print(f"\n=== Training {exp_name} ===")
    print(f"Trainable parameters: {np.sum([np.prod(v.shape) for v in model.trainable_weights])}")
    
    history = model.fit(
        train_gen, 
        validation_data=val_gen, 
        epochs=epochs, 
        callbacks=[ckpt, early, reduce], 
        verbose=1
    )
    return model, history

# -----------------------
# Step 9: Train ResNet-50 First (Memory Efficient)
# -----------------------
print("Building ResNet-50 model...")
res_model = build_enhanced_multitask("ResNet50")
res_model, res_history = compile_and_train(res_model, train_generator, val_generator, "resnet50_generator", epochs=10)

# -----------------------
# Step 10: Evaluate ResNet-50
# -----------------------
print("\nEvaluating ResNet-50...")
res_loss, res_class_loss, res_va_loss, res_class_acc, res_ccc, res_sagr = res_model.evaluate(test_generator)
print(f"ResNet-50 Test Results:")
print(f"Expression Accuracy: {res_class_acc:.4f}")
print(f"Valence-Arousal CCC: {res_ccc:.4f}")
print(f"Valence-Arousal SAGR: {res_sagr:.4f}")

# -----------------------
# Step 11: Train EfficientNet-B0 (Optional - if memory permits)
# -----------------------
try:
    print("\nBuilding EfficientNet-B0 model...")
    eff_model = build_enhanced_multitask("EfficientNetB0")
    eff_model, eff_history = compile_and_train(eff_model, train_generator, val_generator, "efficientnet_generator", epochs=10)
    
    print("\nEvaluating EfficientNet-B0...")
    eff_loss, eff_class_loss, eff_va_loss, eff_class_acc, eff_ccc, eff_sagr = eff_model.evaluate(test_generator)
    print(f"EfficientNet-B0 Test Results:")
    print(f"Expression Accuracy: {eff_class_acc:.4f}")
    print(f"Valence-Arousal CCC: {eff_ccc:.4f}")
    print(f"Valence-Arousal SAGR: {eff_sagr:.4f}")
    
except Exception as e:
    print(f"EfficientNet training skipped due to: {e}")

print("\n=== Training Completed Successfully ===")

# -----------------------
# Step 12: Comparison Results
# -----------------------
print("\n" + "="*50)
print("MODEL COMPARISON RESULTS")
print("="*50)

print(f"\nResNet-50 Performance:")
print(f"- Expression Accuracy: {res_class_acc:.4f}")
print(f"- Valence-Arousal CCC: {res_ccc:.4f}")

try:
    print(f"\nEfficientNet-B0 Performance:")
    print(f"- Expression Accuracy: {eff_class_acc:.4f}")
    print(f"- Valence-Arousal CCC: {eff_ccc:.4f}")
except:
    print(f"\nEfficientNet-B0: Training skipped or failed")

# Recommendation
if res_class_acc > 0.3:
    print(f"\n RECOMMENDATION: Use ResNet-50 (Better performance)")
else:
    print(f"\n RECOMMENDATION: Need model/dataset debugging")